# PySliceKit Example usage
This notebook demonstrates how to use the PySliceKit library to automatically find underperforming segments in both regression and classification models using real-world data.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_california_housing, load_breast_cancer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

import pyslicekit
from pyslicekit.exporter import to_csv, to_json

## 1. Regression Task: California Housing
First, we'll evaluate a Random Forest model on the California Housing dataset.

In [ ]:
# Load data
cali = fetch_california_housing(as_frame=True)
df = cali.frame

# We'll create a mock categorical column just to demonstrate cross-product slicing
np.random.seed(42)
df['ocean_proximity'] = np.random.choice(['INLAND', 'NEAR BAY', 'NEAR OCEAN', '<1H OCEAN'], size=len(df))

X = df.drop(columns=['MedHouseVal'])
y = df['MedHouseVal']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train model
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train.drop(columns=['ocean_proximity']), y_train)

# Make predictions
y_pred = model.predict(X_test.drop(columns=['ocean_proximity']))

In [ ]:
# Evaluate with PySliceKit
results_reg = pyslicekit.evaluate(
    model=model,
    df=X_test,
    y_true=y_test,
    y_pred=y_pred,
    slice_cols=['HouseAge', 'ocean_proximity'],
    metric='mae',
    min_samples=50,
    depth=2,
    render_visuals=True,
    top_n=10
)

### Exporting Results
You can export the results to CSV and JSON formats using the built-in functions.

In [ ]:
to_csv(results_reg, "california_housing_audit.csv")
to_json(results_reg, "california_housing_audit.json")
print("Exported to CSV and JSON successfully!")

## 2. Classification Task: Breast Cancer
Next, we'll evaluate a Logistic Regression model on a classification dataset.

In [ ]:
# Load data
cancer = load_breast_cancer(as_frame=True)
df_c = cancer.frame

X_c = df_c.drop(columns=['target'])
y_c = df_c['target']

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_c, y_c, test_size=0.25, random_state=42)

# Train model
clf = LogisticRegression(max_iter=5000)
clf.fit(X_train_c, y_train_c)

y_pred_c = clf.predict(X_test_c)

In [ ]:
# Evaluate with PySliceKit
results_clf = pyslicekit.evaluate(
    model=clf,
    df=X_test_c,
    y_true=y_test_c,
    y_pred=y_pred_c,
    slice_cols=['mean radius', 'mean texture'],
    metric='f1',
    min_samples=10,
    depth=2,
    render_visuals=True,
    top_n=10
)

In [ ]:
# You can inspect the individual SliceResult objects
worst_segment = results_clf[0]
print(f"Worst segment label: {worst_segment.label}")
print(f"N samples: {worst_segment.n}")
print(f"Metric ({worst_segment.metric_name}): {worst_segment.metric_value:.3f}")
print(f"Gap from overall: {worst_segment.gap:.3f}")
print(f"Is significant? {worst_segment.is_significant} (p-value = {worst_segment.p_value})")